# MSG (Meteosat Second Generation) / SEVIRI (Spinning Enhanced Visible and InfraRed Imager)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=msg/msg-seviri). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L1.5

Level 1.5: calibrated, geolocated and radiometrically corrected image data in 12 channels, in the native (.nat) format, HRIT or NetCDF.

Downloads one full-disc SEVIRI scan from the EUMETSAT Data Store (needs a free account and consumer key/secret, asked at run time) and plots the 10.8 um channel with satpy.

In [ ]:
%pip install -q eumdac satpy netCDF4 h5netcdf matplotlib

In [ ]:
import datetime as dt
import getpass
import shutil
import zipfile
from pathlib import Path

import eumdac

# Free consumer key / secret: https://api.eumetsat.int/api-key (EUMETSAT Data Store account)
token = eumdac.AccessToken((input("EUMETSAT consumer key: "), getpass.getpass("EUMETSAT consumer secret: ")))
datastore = eumdac.DataStore(token)
collection = datastore.get_collection("EO:EUM:DAT:MSG:HRSEVIRI")

products = list(collection.search(dtstart=dt.datetime(2024, 6, 1, 12, 0), dtend=dt.datetime(2024, 6, 1, 12, 15)))
print(len(products), "SEVIRI L1.5 products found")
if not products:
    raise SystemExit("Nothing in this time window - try another one")
product = products[0]
print(product)

out = Path("data")
out.mkdir(exist_ok=True)
# about 100-150 MB per full-disc scan
with product.open() as src, open(out / src.name, "wb") as dst:
    shutil.copyfileobj(src, dst)
for z in out.glob("*.zip"):                     # some collections arrive zipped
    with zipfile.ZipFile(z) as zf:
        zf.extractall(out / z.stem)

from glob import glob

import matplotlib.pyplot as plt
from satpy import Scene

files = sorted(glob("data/**/*.nat", recursive=True))
print(len(files), "files")
scn = Scene(filenames=files, reader="seviri_l1b_native")
print(scn.available_dataset_names())
scn.load(["IR_108"])
scn["IR_108"].plot(cmap="gray_r")              # brightness temperature (K)
plt.title("SEVIRI IR 10.8 um")
plt.show()

## L2

Level 2: EUMETSAT operational products derived from SEVIRI: cloud mask (CLM), cloud top height (CTH), optimal cloud analysis (OCA), global instability index (GII), active fire (FIR), multi-sensor precipitation estimate (MPE), atmospheric motion vectors (AMV), clear-sky radiances (ASR).

Downloads one SEVIRI cloud mask (GRIB2) from the EUMETSAT Data Store and plots the first dataset with satpy. Needs a free account and consumer key/secret, asked at run time.

In [ ]:
%pip install -q eumdac satpy netCDF4 h5netcdf matplotlib eccodes

In [ ]:
import datetime as dt
import getpass
import shutil
import zipfile
from pathlib import Path

import eumdac

# Free consumer key / secret: https://api.eumetsat.int/api-key (EUMETSAT Data Store account)
token = eumdac.AccessToken((input("EUMETSAT consumer key: "), getpass.getpass("EUMETSAT consumer secret: ")))
datastore = eumdac.DataStore(token)
collection = datastore.get_collection("EO:EUM:DAT:MSG:CLM")

products = list(collection.search(dtstart=dt.datetime(2024, 6, 1, 12, 0), dtend=dt.datetime(2024, 6, 1, 12, 30)))
print(len(products), "SEVIRI cloud mask products found")
if not products:
    raise SystemExit("Nothing in this time window - try another one")
product = products[0]
print(product)

out = Path("data")
out.mkdir(exist_ok=True)
# a few MB per product
with product.open() as src, open(out / src.name, "wb") as dst:
    shutil.copyfileobj(src, dst)
for z in out.glob("*.zip"):                     # some collections arrive zipped
    with zipfile.ZipFile(z) as zf:
        zf.extractall(out / z.stem)

from glob import glob

import matplotlib.pyplot as plt
from satpy import Scene

files = sorted(glob("data/**/*.grb*", recursive=True))
print(len(files), "files")
scn = Scene(filenames=files, reader="seviri_l2_grib")
print(scn.available_dataset_names())
name = scn.available_dataset_names()[0]
scn.load([name])
scn[name].plot()
plt.title(name)
plt.show()